In [1]:
import torch
import clip
from PIL import Image

# --------------------------------------------------
# 1. Device
# --------------------------------------------------
device = "mps" if torch.backends.mps.is_available() else "cpu"

# --------------------------------------------------
# 2. Load CLIP ViT-B/32
# --------------------------------------------------
model, preprocess = clip.load("ViT-B/32", device=device)

model.eval()

print("Device:", device)
print("Model: ViT-B/32")

/Users/prathi/work/CLIP/.clipvenv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Device: mps
Model: ViT-B/32


In [2]:
image_path = "/Users/prathi/work/CLIP/learnings/images/National Pet Day Animals.jpg"

image = Image.open(image_path)

print("Original image size:", image.size)
print("Original image mode:", image.mode)

Original image size: (626, 939)
Original image mode: RGB


In [3]:
#apply clip preprocessing to the image

image_tensor = preprocess(image).unsqueeze(0).to(device)
print("After preprocessing:", image_tensor.shape)

After preprocessing: torch.Size([1, 3, 224, 224])


In [4]:
#patchification of the image tensor

visual = model.visual
x = image_tensor.to(visual.conv1.weight.dtype)
print("Input to ViT:", x.shape)
x = visual.conv1(x)
print("After patch projection:", x.shape)

Input to ViT: torch.Size([1, 3, 224, 224])
After patch projection: torch.Size([1, 768, 7, 7])


In [5]:
#convert 7 * 7 patches into 49 patches
x = x.reshape(x.shape[0], x.shape[1], -1)
print("After flattening patches:", x.shape)
x = x.permute(0, 2, 1)
print("Patch tokens:", x.shape)

After flattening patches: torch.Size([1, 768, 49])
Patch tokens: torch.Size([1, 49, 768])


In [6]:
class_token = visual.class_embedding.to(x.dtype)

class_token = class_token + torch.zeros(
    x.shape[0], 1, x.shape[-1],
    dtype=x.dtype,
    device=x.device
)

print("CLS token:", class_token.shape)
x = torch.cat([class_token, x], dim=1)
print("After adding CLS token:", x.shape)

CLS token: torch.Size([1, 1, 768])
After adding CLS token: torch.Size([1, 50, 768])


In [7]:
#positional embedding
x = x + visual.positional_embedding.to(x.dtype)
print("After positional embedding:", x.shape)

After positional embedding: torch.Size([1, 50, 768])


In [ ]:
#OpenAI CLIP uses a LayerNorm before the Transformer
x = visual.ln_pre(x)
print("After ln_pre:", x.shape)

After ln_pre: torch.Size([1, 50, 768])


In [ ]:
#pass through the transformer
x = x.permute(1, 0, 2)
print("Transformer input:", x.shape)
x = visual.transformer(x)
print("Transformer output:", x.shape)

In [9]:
#Get the CLS token from the transformer output
x = x.permute(1, 0, 2)

print("Back to batch-first:", x.shape)

cls_output = x[:, 0, :]

print("CLS output:", cls_output.shape)

Back to batch-first: torch.Size([50, 1, 768])
CLS output: torch.Size([50, 768])


In [10]:
#11 Clip projections

image_embedding = cls_output @ visual.proj

print("Final CLIP image embedding:", image_embedding.shape)

Final CLIP image embedding: torch.Size([50, 512])
